In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

print(f'spark version: {spark.version}')

spark version: 4.1.0


In [0]:
# load the data
df = spark.read.format("json")\
    .option("multiline", True)\
    .option("inferSchema", True)\
    .load('/Volumes/workspace/streaming/streaming/JSON/users.json')

# display the data
display(df)

address,age,city,country,department,email,gender,is_active,join_date,name,orders,phone,salary,skills,user_id
"List(Maadi, 11728, Nile Street)",25,Cairo,Egypt,Data Engineering,ahmed.ali@example.com,Male,true,2023-01-15,Ahmed Ali,"List(List(1001, 25000, Laptop, 1), List(1002, 500, Mouse, 2))",+201001111111,8500,"List(Python, SQL, Spark)",1
"List(Haram, 12511, Pyramids Road)",28,Giza,Egypt,Data Engineering,youssef.hassan@example.com,Male,true,2022-05-20,Youssef Hassan,"List(List(1003, 1200, Keyboard, 1), List(1004, 7000, Monitor, 2))",+201002222222,12000,"List(Python, Kafka, Hadoop, Spark)",2
"List(Stanley, 21500, Corniche Road)",22,Alexandria,Egypt,Marketing,mona.mohamed@example.com,Female,false,2024-02-10,Mona Mohamed,"List(List(1005, 1800, Headphones, 1))",+201003333333,6500,"List(SQL, Excel, Power BI)",3
"List(Downtown, 11511, Tahrir Street)",31,Cairo,Egypt,Data Engineering,omar.khaled@example.com,Male,true,2021-09-01,Omar Khaled,"List(List(1006, 30000, Laptop, 1), List(1007, 3500, SSD, 2), List(1008, 250, USB Cable, 5))",+201004444444,15000,"List(Spark, Scala, Kafka, Flink)",4
"List(Dokki, 12611, Dokki Street)",27,Giza,Egypt,Data Analysis,sara.ahmed@example.com,Female,true,2023-07-18,Sara Ahmed,"List(List(1009, 12000, Tablet, 1), List(1010, 1000, Keyboard, 1))",+201005555555,9500,"List(Python, Pandas, SQL, Power BI)",5
"List(Nasr City, 11765, Nasr City Street)",35,Cairo,Egypt,IT,mahmoud.adel@example.com,Male,false,2020-11-12,Mahmoud Adel,"List(List(1011, 2500, Router, 1))",+201006666666,11000,"List(Hadoop, Hive, Spark)",6
"List(Sidi Gaber, 21523, Sidi Gaber Street)",24,Alexandria,Egypt,Data Analysis,nour.ibrahim@example.com,Female,true,2024-01-05,Nour Ibrahim,"List(List(1012, 18000, Smartphone, 1), List(1013, 900, Power Bank, 2))",+201007777777,7800,"List(Python, Spark, Airflow)",7
"List(Heliopolis, 11341, Heliopolis Street)",29,Cairo,Egypt,Software Engineering,karim.samir@example.com,Male,true,2022-03-14,Karim Samir,"List(List(1014, 6500, Monitor, 2))",+201008888888,13500,"List(Java, Kafka, Flink, Docker)",8
"List(Faisal, 12511, Faisal Street)",26,Giza,Egypt,Business Intelligence,hana.mostafa@example.com,Female,false,2023-10-22,Hana Mostafa,"List(List(1015, 800, Laptop Bag, 1))",+201009999999,7200,"List(SQL, Power BI, Excel)",9
"List(Zamalek, 11211, Zamalek Street)",33,Cairo,Egypt,Data Engineering,tarek.emad@example.com,Male,true,2020-06-30,Tarek Emad,"List(List(1016, 35000, Laptop, 1), List(1017, 6000, Monitor, 3), List(1018, 1200, Keyboard, 2))",+201010101010,16000,"List(Python, Spark, Kafka, Airflow, AWS)",10


In [0]:
# display the schema
df.printSchema()

root
 |-- address: struct (nullable = true)
 |    |-- area: string (nullable = true)
 |    |-- postal_code: string (nullable = true)
 |    |-- street: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- department: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- join_date: string (nullable = true)
 |-- name: string (nullable = true)
 |-- orders: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- order_id: long (nullable = true)
 |    |    |-- price: long (nullable = true)
 |    |    |-- product: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |-- phone: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- user_id: long (nullable = true)



In [0]:
# select specific columns
df.select(
    "user_id", 
    "name", 
    "age", 
    "city", 
    "salary"
).show(truncate=False)

+-------+--------------+---+----------+------+
|user_id|name          |age|city      |salary|
+-------+--------------+---+----------+------+
|1      |Ahmed Ali     |25 |Cairo     |8500  |
|2      |Youssef Hassan|28 |Giza      |12000 |
|3      |Mona Mohamed  |22 |Alexandria|6500  |
|4      |Omar Khaled   |31 |Cairo     |15000 |
|5      |Sara Ahmed    |27 |Giza      |9500  |
|6      |Mahmoud Adel  |35 |Cairo     |11000 |
|7      |Nour Ibrahim  |24 |Alexandria|7800  |
|8      |Karim Samir   |29 |Cairo     |13500 |
|9      |Hana Mostafa  |26 |Giza      |7200  |
|10     |Tarek Emad    |33 |Cairo     |16000 |
|11     |Ali Hassan    |30 |Cairo     |10500 |
|12     |Salma Ahmed   |23 |Alexandria|6800  |
|13     |Hossam Fathy  |36 |Giza      |14500 |
|14     |Dina Sameh    |28 |Cairo     |8200  |
|15     |Mostafa Nabil |32 |Cairo     |13000 |
|16     |Reem Adel     |25 |Giza      |7600  |
|17     |Khaled Mahmoud|38 |Alexandria|17000 |
|18     |Aya Mohamed   |21 |Cairo     |6000  |
|19     |Amr 

In [0]:
# select nested JOSN fields
df.select(
    "name",
    'address.area',
    'address.postal_code'
).show(truncate=False)

+--------------+--------------+-----------+
|name          |area          |postal_code|
+--------------+--------------+-----------+
|Ahmed Ali     |Maadi         |11728      |
|Youssef Hassan|Haram         |12511      |
|Mona Mohamed  |Stanley       |21500      |
|Omar Khaled   |Downtown      |11511      |
|Sara Ahmed    |Dokki         |12611      |
|Mahmoud Adel  |Nasr City     |11765      |
|Nour Ibrahim  |Sidi Gaber    |21523      |
|Karim Samir   |Heliopolis    |11341      |
|Hana Mostafa  |Faisal        |12511      |
|Tarek Emad    |Zamalek       |11211      |
|Ali Hassan    |Abbassia      |11381      |
|Salma Ahmed   |Smouha        |21615      |
|Hossam Fathy  |6th of October|12566      |
|Dina Sameh    |El Shorouk    |11837      |
|Mostafa Nabil |Mokattam      |11571      |
|Reem Adel     |Sheikh Zayed  |12588      |
|Khaled Mahmoud|Miami         |21611      |
|Aya Mohamed   |Garden City   |11519      |
|Amr Sherif    |Agouza        |12654      |
|Laila Hassan  |Nasr City     |1

In [0]:
# rename columns 
# name => full_name
# salary => monthly_salary 
df2 = df.withColumnRenamed(
    "monthly_salary",
    "salary"
).withColumnRenamed(
    "name",
    "full_name"
)

# display data after renaming
df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+
|{Maadi, 11728, Nile Street}               |25 |Cairo     |Egypt  |Data Engineerin

In [0]:
# create the first name and last name
df2 = df2.withColumn(
    "firts_name",
    split(col("full_name"), " ")[0]
).withColumn(
    "last_name",
    split(col("full_name"), " ")[1]
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+
|{Maadi, 11728, Nil

In [0]:
# extract the first name from full name 
df2 = df2.withColumn(
    "firstName",
    split(col('full_name'), " ").getItem(0)
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+

In [0]:
# extract the last name from full name 
df2 = df2.withColumn(
    "lastName",
    split(col('full_name'), " ").getItem(0)
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+---

In [0]:
# create yearly salary column 
df2 = df2.withColumn(
    "yearly_salary",
    col("salary") * 12 
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+--------------------

In [0]:
# create age category 
df2 = df2.withColumn(
    "age_category",
    when(col("age") >= 30, "Junior")
    .otherwise("Senior")
)

# display data after adding age category
df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+-

In [0]:
# create salary category 
df2 = df2.withColumn(
    "salary_category",
    when(col("salary") >= 15000, "High")
    .when(col("salary") >= 10000, "Medium")
    .otherwise("Low")
)

# display data after adding salary category 
df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+---------------------------------------------------------------

In [0]:
# add constant column 
df2 = df2.withColumn(
    "source",
    lit("users_json")
)

# display data after adding the constant 
df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-----------------------------------------

In [0]:
#  find active users 
active_users = df2.filter("is_active == true")

# show active users 
active_users.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-----------------------------------------

In [0]:
# find users older than 30
df2.filter("age > 30")\
    .orderBy(col("age").desc())\
    .show(truncate=False)

+---------------------------------------+---+----------+-------+--------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+
|address                                |age|city      |country|department          |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |
+---------------------------------------+---+----------+-------+--------------------+--------------------------+------+---------+----------+--------------+-----------------------------------------------------

In [0]:
# find users from cairo 
df2.filter(col("city") == 'Cairo')\
    .orderBy(col('city'))\
    .show(truncate=False)

+----------------------------------------+---+-----+-------+---------------------+-------------------------+------+---------+----------+-------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+
|address                                 |age|city |country|department           |email                    |gender|is_active|join_date |full_name    |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |
+----------------------------------------+---+-----+-------+---------------------+-------------------------+------+---------+----------+-------------+--------------------------------------------------------------------

In [0]:
# Find users from Cairo and Giza
df2.filter(
    (col("city") == "Cairo") |
    (col("city") == "Giza")
).orderBy(
    col("city")
).show(truncate=False)

+------------------------------------------+---+-----+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+
|address                                   |age|city |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |
+------------------------------------------+---+-----+-------+---------------------+--------------------------+------+---------+----------+--------------+--------------------------------------------------------

In [0]:
# find users from cairo and giza
df2.filter(
    col("city").isin(["Cairo", "Giza"])
).orderBy(col("city"))\
.show(truncate=False)

+------------------------------------------+---+-----+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+
|address                                   |age|city |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |
+------------------------------------------+---+-----+-------+---------------------+--------------------------+------+---------+----------+--------------+--------------------------------------------------------

In [0]:
# Find active users from Cairo earning more than 10,000.
df2.filter(
    (col("is_active") == True) &
    (col("city").isin("Cairo")) & 
    (col("salary") > 10000)
).show(truncate=False)

+--------------------------------------+---+-----+-------+--------------------+-------------------------+------+---------+----------+-------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+
|address                               |age|city |country|department          |email                    |gender|is_active|join_date |full_name    |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |
+--------------------------------------+---+-----+-------+--------------------+-------------------------+------+---------+----------+-------------+-----------------------------------------------------------------------------

In [0]:
# sort salary ascending 
df2.orderBy(col("salary").asc())\
    .show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-----------------------------------------

In [0]:
# sort by department and salary
df2.orderBy(
    col("department"),
    col("salary").desc()
).show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-----------------------------------------

In [0]:
# convert names to upper case
df2 = df2.withColumn(
    'name_upper',
    upper(col("full_name"))
)

# display data after transforming the name
df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-----------

In [0]:
# convert names to lower case
df2 = df2.withColumn(
    "name_lower",
    lower(col("full_name"))
)

# show data after transforming
df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |
+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+-------

In [0]:
# calculate the full name length 
df2 = df2.withColumn(
    "name_length",
    length(col("full_name"))
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|
+------------------------------------------+---+----------+-------+---------------------+--------------------------+

In [0]:
# replace part of an email 
df2 = df2.withColumn(
    "updated_email",
    regexp_replace(col("email"), "@example.com", "@gmail.com")
)

# show data after replace the characters
df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|updated_email           |
+------------------------------------------+---+----------+-------

In [0]:
# create username from email 
df2 = df2.withColumn(
    'username',
    split("email", "@").getItem(0)
)

# show new data 
df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|updated_email           |username      |
+-----------------------------------

In [0]:
# extract joining year 
df2 = df2.withColumn(
    'join_year',
    year(col("join_date"))
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|updated_email           |username      |join_year|
+---------------

In [0]:
# extract join_month 
df2 = df2.withColumn(
    "join_month",
    month(col("join_date"))
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|updated_email           |username      |join_year|join_m

In [0]:
# calculate years since joining 
df2 = df2.withColumn(
    "years_since_joininag",
    year(current_date()) - year(col('join_date'))
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|updated_email           |username  

In [0]:
# calculate the number of months
df2 = df2.withColumn(
    "months_since_joining",
    months_between(
        current_date(),
        col("join_date")
    ).cast("int")
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|updated_email 

In [0]:
# count users - show number of users 
df2.count()

20

In [0]:
# count the number of columns 
print(f"number of columns: {len(df2.columns)}")

number of columns: 32


In [0]:
# find the average salary 
df2.select(
    avg(col('salary')).cast('int').alias('average_salary')
).show()

+--------------+
|average_salary|
+--------------+
|         10645|
+--------------+



In [0]:
# find the min salary and max salary
df2.select(
    min(col('salary')).alias("min_salary"),
    max(col('salary')).alias('max_salary')
).show()

+----------+----------+
|min_salary|max_salary|
+----------+----------+
|      6000|     17000|
+----------+----------+



In [0]:
# find the total salaries
df2.select(
    sum(col('salary')).alias('total_salaries')
).show()

+--------------+
|total_salaries|
+--------------+
|        212900|
+--------------+



In [0]:
# count users per city and calculate the total salaries
df2.groupBy('city')\
    .agg(
        count('*').alias('customers_count'),
        sum(col('salary')).alias('total_salary')
        ).show()

+----------+---------------+------------+
|      city|customers_count|total_salary|
+----------+---------------+------------+
|      Giza|              6|       63300|
|     Cairo|             10|      111500|
|Alexandria|              4|       38100|
+----------+---------------+------------+



In [0]:
# count users per city 
df2.groupBy("city")\
    .agg(count("*").alias('users_count'))\
    .show(truncate=False)

+----------+-----------+
|city      |users_count|
+----------+-----------+
|Giza      |6          |
|Cairo     |10         |
|Alexandria|4          |
+----------+-----------+



In [0]:
# average salary per department
df2.groupBy("department")\
    .agg(avg("salary").alias("average_salary").cast("int"))\
    .show(truncate=False)

+---------------------+--------------+
|department           |average_salary|
+---------------------+--------------+
|Data Analysis        |8300          |
|IT                   |11333         |
|Marketing            |6433          |
|HR                   |8200          |
|Data Engineering     |13833         |
|Software Engineering |13250         |
|Business Intelligence|8500          |
+---------------------+--------------+



In [0]:
# multiple aggregations
df2.groupBy("department")\
    .agg(
        count('*').alias('users_count'),
        sum("salary").alias("total_salaries"),
        avg("salary").alias("average_salary").cast("int"),
        min("salary").alias("min_salary"),
        max("salary").alias("max_salary")
    ).show(truncate=False)

+---------------------+-----------+--------------+--------------+----------+----------+
|department           |users_count|total_salaries|average_salary|min_salary|max_salary|
+---------------------+-----------+--------------+--------------+----------+----------+
|Data Analysis        |3          |24900         |8300          |7600      |9500      |
|IT                   |3          |34000         |11333         |10500     |12500     |
|Marketing            |3          |19300         |6433          |6000      |6800      |
|HR                   |1          |8200          |8200          |8200      |8200      |
|Data Engineering     |6          |83000         |13833         |8500      |17000     |
|Software Engineering |2          |26500         |13250         |13000     |13500     |
|Business Intelligence|2          |17000         |8500          |7200      |9800      |
+---------------------+-----------+--------------+--------------+----------+----------+



In [0]:
# find departments whose average salary is greater than 10000
df2.groupBy("department")\
    .agg(avg("salary").cast("int").alias("average_salary"))\
    .filter(col("average_salary") > 10000)\
    .show(truncate=False)

+--------------------+--------------+
|department          |average_salary|
+--------------------+--------------+
|IT                  |11333         |
|Data Engineering    |13833         |
|Software Engineering|13250         |
+--------------------+--------------+



In [0]:
# count nulls in every column
df2.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df2.columns
]).show(truncate=False)

+-------+---+----+-------+----------+-----+------+---------+---------+---------+------+-----+------+------+-------+----------+---------+---------+--------+-------------+------------+---------------+------+----------+----------+-----------+-------------+--------+---------+----------+--------------------+--------------------+
|address|age|city|country|department|email|gender|is_active|join_date|full_name|orders|phone|salary|skills|user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source|name_upper|name_lower|name_length|updated_email|username|join_year|join_month|years_since_joininag|months_since_joining|
+-------+---+----+-------+----------+-----+------+---------+---------+---------+------+-----+------+------+-------+----------+---------+---------+--------+-------------+------------+---------------+------+----------+----------+-----------+-------------+--------+---------+----------+--------------------+--------------------+
|0      |0  |0   |0   

In [0]:
df2.dropna().show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|updated_email 

In [0]:
# fill Nulls 
df2.fillna({
    "salary": 0,
    "city": "unknown"
}).show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|updated_email 

In [0]:
# remove duplicates
df2.dropDuplicates().show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |name_lower    |name_length|updated_email 

# JSON Transformations

In [0]:
# select specific columns 
df2.select("full_name", "skills").show(truncate=False)

+--------------+------------------------------------+
|full_name     |skills                              |
+--------------+------------------------------------+
|Ahmed Ali     |[Python, SQL, Spark]                |
|Youssef Hassan|[Python, Kafka, Hadoop, Spark]      |
|Mona Mohamed  |[SQL, Excel, Power BI]              |
|Omar Khaled   |[Spark, Scala, Kafka, Flink]        |
|Sara Ahmed    |[Python, Pandas, SQL, Power BI]     |
|Mahmoud Adel  |[Hadoop, Hive, Spark]               |
|Nour Ibrahim  |[Python, Spark, Airflow]            |
|Karim Samir   |[Java, Kafka, Flink, Docker]        |
|Hana Mostafa  |[SQL, Power BI, Excel]              |
|Tarek Emad    |[Python, Spark, Kafka, Airflow, AWS]|
|Ali Hassan    |[Linux, Docker, Kubernetes]         |
|Salma Ahmed   |[Excel, SQL, Power BI]              |
|Hossam Fathy  |[Python, Spark, Hadoop, Hive]       |
|Dina Sameh    |[Excel, SQL]                        |
|Mostafa Nabil |[Python, Django, Docker, Git]       |
|Reem Adel     |[Python, Pan

In [0]:
# get first, second skill
df2.select(
    "full_name", 
    col("skills")[0].alias("first_skill"),
    col("skills")[1].alias("second_skill")
).show(truncate=False)

+--------------+-----------+------------+
|full_name     |first_skill|second_skill|
+--------------+-----------+------------+
|Ahmed Ali     |Python     |SQL         |
|Youssef Hassan|Python     |Kafka       |
|Mona Mohamed  |SQL        |Excel       |
|Omar Khaled   |Spark      |Scala       |
|Sara Ahmed    |Python     |Pandas      |
|Mahmoud Adel  |Hadoop     |Hive        |
|Nour Ibrahim  |Python     |Spark       |
|Karim Samir   |Java       |Kafka       |
|Hana Mostafa  |SQL        |Power BI    |
|Tarek Emad    |Python     |Spark       |
|Ali Hassan    |Linux      |Docker      |
|Salma Ahmed   |Excel      |SQL         |
|Hossam Fathy  |Python     |Spark       |
|Dina Sameh    |Excel      |SQL         |
|Mostafa Nabil |Python     |Django      |
|Reem Adel     |Python     |Pandas      |
|Khaled Mahmoud|Spark      |Scala       |
|Aya Mohamed   |Excel      |Power BI    |
|Amr Sherif    |Linux      |Hadoop      |
|Laila Hassan  |SQL        |Power BI    |
+--------------+-----------+------

In [0]:
# get number of skills
df2.select(
    "full_name",
    size("skills").alias("number_of_skills")
).show(truncate=False)

+--------------+----------------+
|full_name     |number_of_skills|
+--------------+----------------+
|Ahmed Ali     |3               |
|Youssef Hassan|4               |
|Mona Mohamed  |3               |
|Omar Khaled   |4               |
|Sara Ahmed    |4               |
|Mahmoud Adel  |3               |
|Nour Ibrahim  |3               |
|Karim Samir   |4               |
|Hana Mostafa  |3               |
|Tarek Emad    |5               |
|Ali Hassan    |3               |
|Salma Ahmed   |3               |
|Hossam Fathy  |4               |
|Dina Sameh    |2               |
|Mostafa Nabil |4               |
|Reem Adel     |3               |
|Khaled Mahmoud|5               |
|Aya Mohamed   |3               |
|Amr Sherif    |4               |
|Laila Hassan  |4               |
+--------------+----------------+



In [0]:
# check whether use knows python
df2.select(
    "full_name",
    array_contains("skills", "python").alias("knows_python")
).show(truncate=False)

+--------------+------------+
|full_name     |knows_python|
+--------------+------------+
|Ahmed Ali     |false       |
|Youssef Hassan|false       |
|Mona Mohamed  |false       |
|Omar Khaled   |false       |
|Sara Ahmed    |false       |
|Mahmoud Adel  |false       |
|Nour Ibrahim  |false       |
|Karim Samir   |false       |
|Hana Mostafa  |false       |
|Tarek Emad    |false       |
|Ali Hassan    |false       |
|Salma Ahmed   |false       |
|Hossam Fathy  |false       |
|Dina Sameh    |false       |
|Mostafa Nabil |false       |
|Reem Adel     |false       |
|Khaled Mahmoud|false       |
|Aya Mohamed   |false       |
|Amr Sherif    |false       |
|Laila Hassan  |false       |
+--------------+------------+



In [0]:
# explode skills
df2.select(
    "full_name",
    explode("skills").alias("skill")
).show(truncate=False)

+--------------+--------+
|full_name     |skill   |
+--------------+--------+
|Ahmed Ali     |Python  |
|Ahmed Ali     |SQL     |
|Ahmed Ali     |Spark   |
|Youssef Hassan|Python  |
|Youssef Hassan|Kafka   |
|Youssef Hassan|Hadoop  |
|Youssef Hassan|Spark   |
|Mona Mohamed  |SQL     |
|Mona Mohamed  |Excel   |
|Mona Mohamed  |Power BI|
|Omar Khaled   |Spark   |
|Omar Khaled   |Scala   |
|Omar Khaled   |Kafka   |
|Omar Khaled   |Flink   |
|Sara Ahmed    |Python  |
|Sara Ahmed    |Pandas  |
|Sara Ahmed    |SQL     |
|Sara Ahmed    |Power BI|
|Mahmoud Adel  |Hadoop  |
|Mahmoud Adel  |Hive    |
+--------------+--------+
only showing top 20 rows


In [0]:
df2.select(
    "full_name",
    explode_outer("skills").alias("skill")
).show(truncate=False)

+--------------+--------+
|full_name     |skill   |
+--------------+--------+
|Ahmed Ali     |Python  |
|Ahmed Ali     |SQL     |
|Ahmed Ali     |Spark   |
|Youssef Hassan|Python  |
|Youssef Hassan|Kafka   |
|Youssef Hassan|Hadoop  |
|Youssef Hassan|Spark   |
|Mona Mohamed  |SQL     |
|Mona Mohamed  |Excel   |
|Mona Mohamed  |Power BI|
|Omar Khaled   |Spark   |
|Omar Khaled   |Scala   |
|Omar Khaled   |Kafka   |
|Omar Khaled   |Flink   |
|Sara Ahmed    |Python  |
|Sara Ahmed    |Pandas  |
|Sara Ahmed    |SQL     |
|Sara Ahmed    |Power BI|
|Mahmoud Adel  |Hadoop  |
|Mahmoud Adel  |Hive    |
+--------------+--------+
only showing top 20 rows


# Dealing With Nested JSON Objects

In [0]:
# select the entire address
df2.select(
    "full_name",
    'address'
).show(truncate=False)

+--------------+------------------------------------------+
|full_name     |address                                   |
+--------------+------------------------------------------+
|Ahmed Ali     |{Maadi, 11728, Nile Street}               |
|Youssef Hassan|{Haram, 12511, Pyramids Road}             |
|Mona Mohamed  |{Stanley, 21500, Corniche Road}           |
|Omar Khaled   |{Downtown, 11511, Tahrir Street}          |
|Sara Ahmed    |{Dokki, 12611, Dokki Street}              |
|Mahmoud Adel  |{Nasr City, 11765, Nasr City Street}      |
|Nour Ibrahim  |{Sidi Gaber, 21523, Sidi Gaber Street}    |
|Karim Samir   |{Heliopolis, 11341, Heliopolis Street}    |
|Hana Mostafa  |{Faisal, 12511, Faisal Street}            |
|Tarek Emad    |{Zamalek, 11211, Zamalek Street}          |
|Ali Hassan    |{Abbassia, 11381, Abbassia Street}        |
|Salma Ahmed   |{Smouha, 21615, Smouha Street}            |
|Hossam Fathy  |{6th of October, 12566, October Street}   |
|Dina Sameh    |{El Shorouk, 11837, Shor

In [0]:
# select individual nested fields
df2.select(
    "full_name",
    "address.street",
    "address.area",
    "address.postal_code"
).show(truncate=False)

+--------------+-------------------+--------------+-----------+
|full_name     |street             |area          |postal_code|
+--------------+-------------------+--------------+-----------+
|Ahmed Ali     |Nile Street        |Maadi         |11728      |
|Youssef Hassan|Pyramids Road      |Haram         |12511      |
|Mona Mohamed  |Corniche Road      |Stanley       |21500      |
|Omar Khaled   |Tahrir Street      |Downtown      |11511      |
|Sara Ahmed    |Dokki Street       |Dokki         |12611      |
|Mahmoud Adel  |Nasr City Street   |Nasr City     |11765      |
|Nour Ibrahim  |Sidi Gaber Street  |Sidi Gaber    |21523      |
|Karim Samir   |Heliopolis Street  |Heliopolis    |11341      |
|Hana Mostafa  |Faisal Street      |Faisal        |12511      |
|Tarek Emad    |Zamalek Street     |Zamalek       |11211      |
|Ali Hassan    |Abbassia Street    |Abbassia      |11381      |
|Salma Ahmed   |Smouha Street      |Smouha        |21615      |
|Hossam Fathy  |October Street     |6th 

In [0]:
# create a full address
df2 = df2.withColumn(
    "full_address",
    concat_ws(' ', "address.street", "address.area", "address.postal_code")
)

df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+--------------------------------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |na

# Dealing with Nested Arrays

In [0]:
# display orders 
df2.select(
    "full_name",
    'orders'
).show(truncate=False)

+--------------+-------------------------------------------------------------------------------+
|full_name     |orders                                                                         |
+--------------+-------------------------------------------------------------------------------+
|Ahmed Ali     |[{1001, 25000, Laptop, 1}, {1002, 500, Mouse, 2}]                              |
|Youssef Hassan|[{1003, 1200, Keyboard, 1}, {1004, 7000, Monitor, 2}]                          |
|Mona Mohamed  |[{1005, 1800, Headphones, 1}]                                                  |
|Omar Khaled   |[{1006, 30000, Laptop, 1}, {1007, 3500, SSD, 2}, {1008, 250, USB Cable, 5}]    |
|Sara Ahmed    |[{1009, 12000, Tablet, 1}, {1010, 1000, Keyboard, 1}]                          |
|Mahmoud Adel  |[{1011, 2500, Router, 1}]                                                      |
|Nour Ibrahim  |[{1012, 18000, Smartphone, 1}, {1013, 900, Power Bank, 2}]                     |
|Karim Samir   |[{1014, 6500, 

In [0]:
# get first order's product
df2.select(
    "full_name",
    col("orders")[0]['product'].alias('product')
).show(truncate=False)

+--------------+-------------------+
|full_name     |product            |
+--------------+-------------------+
|Ahmed Ali     |Laptop             |
|Youssef Hassan|Keyboard           |
|Mona Mohamed  |Headphones         |
|Omar Khaled   |Laptop             |
|Sara Ahmed    |Tablet             |
|Mahmoud Adel  |Router             |
|Nour Ibrahim  |Smartphone         |
|Karim Samir   |Monitor            |
|Hana Mostafa  |Laptop Bag         |
|Tarek Emad    |Laptop             |
|Ali Hassan    |Server             |
|Salma Ahmed   |Tablet             |
|Hossam Fathy  |Laptop             |
|Dina Sameh    |Headphones         |
|Mostafa Nabil |Mechanical Keyboard|
|Reem Adel     |Laptop             |
|Khaled Mahmoud|Server             |
|Aya Mohamed   |Tablet             |
|Amr Sherif    |Router             |
|Laila Hassan  |Monitor            |
+--------------+-------------------+



In [0]:
# count orders 
df2.select(
    'full_name',
    size('orders').alias('orders_count')
).show(truncate=False)

+--------------+------------+
|full_name     |orders_count|
+--------------+------------+
|Ahmed Ali     |2           |
|Youssef Hassan|2           |
|Mona Mohamed  |1           |
|Omar Khaled   |3           |
|Sara Ahmed    |2           |
|Mahmoud Adel  |1           |
|Nour Ibrahim  |2           |
|Karim Samir   |1           |
|Hana Mostafa  |1           |
|Tarek Emad    |3           |
|Ali Hassan    |1           |
|Salma Ahmed   |1           |
|Hossam Fathy  |2           |
|Dina Sameh    |1           |
|Mostafa Nabil |2           |
|Reem Adel     |1           |
|Khaled Mahmoud|2           |
|Aya Mohamed   |1           |
|Amr Sherif    |1           |
|Laila Hassan  |2           |
+--------------+------------+



In [0]:
# explodeorders 
orders_df = df2.select(
    'full_name',
    explode('orders').alias('orders')
)

orders_df.show(truncate=False)

+--------------+----------------------------+
|full_name     |orders                      |
+--------------+----------------------------+
|Ahmed Ali     |{1001, 25000, Laptop, 1}    |
|Ahmed Ali     |{1002, 500, Mouse, 2}       |
|Youssef Hassan|{1003, 1200, Keyboard, 1}   |
|Youssef Hassan|{1004, 7000, Monitor, 2}    |
|Mona Mohamed  |{1005, 1800, Headphones, 1} |
|Omar Khaled   |{1006, 30000, Laptop, 1}    |
|Omar Khaled   |{1007, 3500, SSD, 2}        |
|Omar Khaled   |{1008, 250, USB Cable, 5}   |
|Sara Ahmed    |{1009, 12000, Tablet, 1}    |
|Sara Ahmed    |{1010, 1000, Keyboard, 1}   |
|Mahmoud Adel  |{1011, 2500, Router, 1}     |
|Nour Ibrahim  |{1012, 18000, Smartphone, 1}|
|Nour Ibrahim  |{1013, 900, Power Bank, 2}  |
|Karim Samir   |{1014, 6500, Monitor, 2}    |
|Hana Mostafa  |{1015, 800, Laptop Bag, 1}  |
|Tarek Emad    |{1016, 35000, Laptop, 1}    |
|Tarek Emad    |{1017, 6000, Monitor, 3}    |
|Tarek Emad    |{1018, 1200, Keyboard, 2}   |
|Ali Hassan    |{1019, 45000, Serv

In [0]:
# check the schema of the df
df2.printSchema()

root
 |-- address: struct (nullable = true)
 |    |-- area: string (nullable = true)
 |    |-- postal_code: string (nullable = true)
 |    |-- street: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- department: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- join_date: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- orders: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- order_id: long (nullable = true)
 |    |    |-- price: long (nullable = true)
 |    |    |-- product: string (nullable = true)
 |    |    |-- quantity: long (nullable = true)
 |-- phone: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- user_id: long (nullable = true)
 |-- 

In [0]:
# extract order fields

# first explode all orders 
orders_df = df2.select(
    "full_name",
    explode("orders").alias("order")
)

# second explode all fields
orders_df = orders_df.select(
    "full_name",
    col("order.order_id").alias("order_id"),
    col("order.product").alias("product"),
    col("order.quantity").alias("quantity"),
    col("order.price").alias("price")
)

# print the final result
orders_df.show(truncate=False)

+--------------+--------+----------+--------+-----+
|full_name     |order_id|product   |quantity|price|
+--------------+--------+----------+--------+-----+
|Ahmed Ali     |1001    |Laptop    |1       |25000|
|Ahmed Ali     |1002    |Mouse     |2       |500  |
|Youssef Hassan|1003    |Keyboard  |1       |1200 |
|Youssef Hassan|1004    |Monitor   |2       |7000 |
|Mona Mohamed  |1005    |Headphones|1       |1800 |
|Omar Khaled   |1006    |Laptop    |1       |30000|
|Omar Khaled   |1007    |SSD       |2       |3500 |
|Omar Khaled   |1008    |USB Cable |5       |250  |
|Sara Ahmed    |1009    |Tablet    |1       |12000|
|Sara Ahmed    |1010    |Keyboard  |1       |1000 |
|Mahmoud Adel  |1011    |Router    |1       |2500 |
|Nour Ibrahim  |1012    |Smartphone|1       |18000|
|Nour Ibrahim  |1013    |Power Bank|2       |900  |
|Karim Samir   |1014    |Monitor   |2       |6500 |
|Hana Mostafa  |1015    |Laptop Bag|1       |800  |
|Tarek Emad    |1016    |Laptop    |1       |35000|
|Tarek Emad 

In [0]:
# calculate the order value 
orders_df = orders_df.withColumn(
    'total',
    col("price") * col('quantity')
)

orders_df.show()

+--------------+--------+----------+--------+-----+-----+
|     full_name|order_id|   product|quantity|price|total|
+--------------+--------+----------+--------+-----+-----+
|     Ahmed Ali|    1001|    Laptop|       1|25000|25000|
|     Ahmed Ali|    1002|     Mouse|       2|  500| 1000|
|Youssef Hassan|    1003|  Keyboard|       1| 1200| 1200|
|Youssef Hassan|    1004|   Monitor|       2| 7000|14000|
|  Mona Mohamed|    1005|Headphones|       1| 1800| 1800|
|   Omar Khaled|    1006|    Laptop|       1|30000|30000|
|   Omar Khaled|    1007|       SSD|       2| 3500| 7000|
|   Omar Khaled|    1008| USB Cable|       5|  250| 1250|
|    Sara Ahmed|    1009|    Tablet|       1|12000|12000|
|    Sara Ahmed|    1010|  Keyboard|       1| 1000| 1000|
|  Mahmoud Adel|    1011|    Router|       1| 2500| 2500|
|  Nour Ibrahim|    1012|Smartphone|       1|18000|18000|
|  Nour Ibrahim|    1013|Power Bank|       2|  900| 1800|
|   Karim Samir|    1014|   Monitor|       2| 6500|13000|
|  Hana Mostaf

In [0]:
# find expensive orders 
orders_df.filter(
    col('total') > 10000
).orderBy(col('total').desc())\
    .show(truncate=False)

+--------------+--------+----------+--------+-----+------+
|full_name     |order_id|product   |quantity|price|total |
+--------------+--------+----------+--------+-----+------+
|Khaled Mahmoud|1027    |Server    |2       |50000|100000|
|Hossam Fathy  |1021    |Laptop    |2       |28000|56000 |
|Ali Hassan    |1019    |Server    |1       |45000|45000 |
|Tarek Emad    |1016    |Laptop    |1       |35000|35000 |
|Omar Khaled   |1006    |Laptop    |1       |30000|30000 |
|Ahmed Ali     |1001    |Laptop    |1       |25000|25000 |
|Reem Adel     |1026    |Laptop    |1       |22000|22000 |
|Tarek Emad    |1017    |Monitor   |3       |6000 |18000 |
|Nour Ibrahim  |1012    |Smartphone|1       |18000|18000 |
|Khaled Mahmoud|1028    |SSD       |5       |3200 |16000 |
|Youssef Hassan|1004    |Monitor   |2       |7000 |14000 |
|Karim Samir   |1014    |Monitor   |2       |6500 |13000 |
|Sara Ahmed    |1009    |Tablet    |1       |12000|12000 |
|Salma Ahmed   |1020    |Tablet    |1       |11000|11000

In [0]:
# calculate the total spending per user 
orders_df.groupBy("full_name") \
    .agg(
        sum(col("quantity") * col("price")).alias("total_spend")
    ) \
    .orderBy(col("total_spend").desc()) \
    .show(truncate=False)

+--------------+-----------+
|full_name     |total_spend|
+--------------+-----------+
|Khaled Mahmoud|116000     |
|Hossam Fathy  |65000      |
|Tarek Emad    |55400      |
|Ali Hassan    |45000      |
|Omar Khaled   |38250      |
|Ahmed Ali     |26000      |
|Reem Adel     |22000      |
|Nour Ibrahim  |19800      |
|Youssef Hassan|15200      |
|Karim Samir   |13000      |
|Sara Ahmed    |13000      |
|Salma Ahmed   |11000      |
|Aya Mohamed   |9500       |
|Laila Hassan  |8600       |
|Amr Sherif    |6000       |
|Mostafa Nabil |4200       |
|Dina Sameh    |3200       |
|Mahmoud Adel  |2500       |
|Mona Mohamed  |1800       |
|Hana Mostafa  |800        |
+--------------+-----------+



# Array Functions

In [0]:
# sort skills 
df2.select(
    'full_name',
    sort_array('skills').alias("sorted_skills")
).show(truncate=False)

+--------------+------------------------------------+
|full_name     |sorted_skills                       |
+--------------+------------------------------------+
|Ahmed Ali     |[Python, SQL, Spark]                |
|Youssef Hassan|[Hadoop, Kafka, Python, Spark]      |
|Mona Mohamed  |[Excel, Power BI, SQL]              |
|Omar Khaled   |[Flink, Kafka, Scala, Spark]        |
|Sara Ahmed    |[Pandas, Power BI, Python, SQL]     |
|Mahmoud Adel  |[Hadoop, Hive, Spark]               |
|Nour Ibrahim  |[Airflow, Python, Spark]            |
|Karim Samir   |[Docker, Flink, Java, Kafka]        |
|Hana Mostafa  |[Excel, Power BI, SQL]              |
|Tarek Emad    |[AWS, Airflow, Kafka, Python, Spark]|
|Ali Hassan    |[Docker, Kubernetes, Linux]         |
|Salma Ahmed   |[Excel, Power BI, SQL]              |
|Hossam Fathy  |[Hadoop, Hive, Python, Spark]       |
|Dina Sameh    |[Excel, SQL]                        |
|Mostafa Nabil |[Django, Docker, Git, Python]       |
|Reem Adel     |[NumPy, Pand

In [0]:
# reverse sorting 
df2.select(
    "full_name",
    reverse('skills').alias('reverse_skills')
).show(truncate=False)

+--------------+------------------------------------+
|full_name     |reverse_skills                      |
+--------------+------------------------------------+
|Ahmed Ali     |[Spark, SQL, Python]                |
|Youssef Hassan|[Spark, Hadoop, Kafka, Python]      |
|Mona Mohamed  |[Power BI, Excel, SQL]              |
|Omar Khaled   |[Flink, Kafka, Scala, Spark]        |
|Sara Ahmed    |[Power BI, SQL, Pandas, Python]     |
|Mahmoud Adel  |[Spark, Hive, Hadoop]               |
|Nour Ibrahim  |[Airflow, Spark, Python]            |
|Karim Samir   |[Docker, Flink, Kafka, Java]        |
|Hana Mostafa  |[Excel, Power BI, SQL]              |
|Tarek Emad    |[AWS, Airflow, Kafka, Spark, Python]|
|Ali Hassan    |[Kubernetes, Docker, Linux]         |
|Salma Ahmed   |[Power BI, SQL, Excel]              |
|Hossam Fathy  |[Hive, Hadoop, Spark, Python]       |
|Dina Sameh    |[SQL, Excel]                        |
|Mostafa Nabil |[Git, Docker, Django, Python]       |
|Reem Adel     |[NumPy, Pand

In [0]:
# join skills into string 
df2.select(
    'full_name',
    concat_ws(", ", 'skills').alias('skills')
).show(truncate=False)

+--------------+----------------------------------+
|full_name     |skills                            |
+--------------+----------------------------------+
|Ahmed Ali     |Python, SQL, Spark                |
|Youssef Hassan|Python, Kafka, Hadoop, Spark      |
|Mona Mohamed  |SQL, Excel, Power BI              |
|Omar Khaled   |Spark, Scala, Kafka, Flink        |
|Sara Ahmed    |Python, Pandas, SQL, Power BI     |
|Mahmoud Adel  |Hadoop, Hive, Spark               |
|Nour Ibrahim  |Python, Spark, Airflow            |
|Karim Samir   |Java, Kafka, Flink, Docker        |
|Hana Mostafa  |SQL, Power BI, Excel              |
|Tarek Emad    |Python, Spark, Kafka, Airflow, AWS|
|Ali Hassan    |Linux, Docker, Kubernetes         |
|Salma Ahmed   |Excel, SQL, Power BI              |
|Hossam Fathy  |Python, Spark, Hadoop, Hive       |
|Dina Sameh    |Excel, SQL                        |
|Mostafa Nabil |Python, Django, Docker, Git       |
|Reem Adel     |Python, Pandas, NumPy             |
|Khaled Mahm

# JSON Functions

In [0]:
# convert an address struct to json 
df2.select(
    'full_name',
    to_json('address').alias('address_json')
).show(truncate=False)

+--------------+----------------------------------------------------------------------------+
|full_name     |address_json                                                                |
+--------------+----------------------------------------------------------------------------+
|Ahmed Ali     |{"area":"Maadi","postal_code":"11728","street":"Nile Street"}               |
|Youssef Hassan|{"area":"Haram","postal_code":"12511","street":"Pyramids Road"}             |
|Mona Mohamed  |{"area":"Stanley","postal_code":"21500","street":"Corniche Road"}           |
|Omar Khaled   |{"area":"Downtown","postal_code":"11511","street":"Tahrir Street"}          |
|Sara Ahmed    |{"area":"Dokki","postal_code":"12611","street":"Dokki Street"}              |
|Mahmoud Adel  |{"area":"Nasr City","postal_code":"11765","street":"Nasr City Street"}      |
|Nour Ibrahim  |{"area":"Sidi Gaber","postal_code":"21523","street":"Sidi Gaber Street"}    |
|Karim Samir   |{"area":"Heliopolis","postal_code":"11341","

In [0]:
# convert orders to json 
df2.select(
    'full_name',
    to_json('orders').alias('orders_json')
).show(truncate=False)

+--------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|full_name     |orders_json                                                                                                                                                                                       |
+--------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Ahmed Ali     |[{"order_id":1001,"price":25000,"product":"Laptop","quantity":1},{"order_id":1002,"price":500,"product":"Mouse","quantity":2}]                                                                    |
|Youssef Hassan|[{"order_id":1003,"price":1200,"product":"Keyboard","quantity":1},{"order_id":1004,"price":7000,"product":"Monitor","quantity":2}]      

# `from_json()` — Parsing JSON Strings into Structured Data

## Kafka + Spark JSON Parsing Pattern

This is the common pattern you'll use when working with **Kafka + Spark Structured Streaming**:
<hr>

```text
Kafka `value`
      ↓
`cast("string")`
      ↓
`from_json()`
      ↓
`StructType`
      ↓
`select("data.*")`

In [0]:
# Create a DataFrame containing JSON strings
json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","age":25}',),
    ('{"id":102,"name":"Youssef","age":28}',)
], ["json_string"])

# Define schema
schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("age", IntegerType())
])

# Parse JSON string into a struct
parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.show(truncate=False)

# Extract fields from the struct
parsed_df.select(
    "data.id",
    "data.name",
    "data.age"
).show(truncate=False)

+------------------------------------+------------------+
|json_string                         |data              |
+------------------------------------+------------------+
|{"id":101,"name":"Ahmed","age":25}  |{101, Ahmed, 25}  |
|{"id":102,"name":"Youssef","age":28}|{102, Youssef, 28}|
+------------------------------------+------------------+

+---+-------+---+
|id |name   |age|
+---+-------+---+
|101|Ahmed  |25 |
|102|Youssef|28 |
+---+-------+---+



In [0]:
# ============================================================
# Task 1: Parse basic JSON strings and extract id, name, age.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","age":25}',),
    ('{"id":102,"name":"Youssef","age":28}',)
], ["json_string"])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("age", IntegerType())
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select("data.*").show(truncate=False)

+---+-------+---+
|id |name   |age|
+---+-------+---+
|101|Ahmed  |25 |
|102|Youssef|28 |
+---+-------+---+



In [0]:
# ============================================================
# Task 2: Parse nested JSON and extract city and country.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","address":{"city":"Cairo","country":"Egypt"}}',),
    ('{"id":102,"name":"Youssef","address":{"city":"Giza","country":"Egypt"}}',)
], ["json_string"])

address_schema = StructType([
    StructField("city", StringType()),
    StructField("country", StringType())
])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("address", address_schema)
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select(
    "data.id",
    "data.name",
    "data.address.city",
    "data.address.country"
).show(truncate=False)

+---+-------+-----+-------+
|id |name   |city |country|
+---+-------+-----+-------+
|101|Ahmed  |Cairo|Egypt  |
|102|Youssef|Giza |Egypt  |
+---+-------+-----+-------+



In [0]:
# ============================================================
# Task 3: Parse JSON containing an array of skills.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","skills":["Python","SQL","Spark"]}',),
    ('{"id":102,"name":"Youssef","skills":["Python","Kafka","Spark"]}',)
], ["json_string"])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("skills", ArrayType(StringType()))
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select("data.*").show(truncate=False)


+---+-------+----------------------+
|id |name   |skills                |
+---+-------+----------------------+
|101|Ahmed  |[Python, SQL, Spark]  |
|102|Youssef|[Python, Kafka, Spark]|
+---+-------+----------------------+



In [0]:
# ============================================================
# Task 4: Parse JSON containing an array of orders and explode it.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","orders":[{"order_id":1,"product":"Laptop","quantity":1,"price":1000},{"order_id":2,"product":"Mouse","quantity":2,"price":50}]}',),
    ('{"id":102,"name":"Youssef","orders":[{"order_id":3,"product":"Keyboard","quantity":1,"price":100}]}',)
], ["json_string"])

order_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("product", StringType()),
    StructField("quantity", IntegerType()),
    StructField("price", IntegerType())
])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("orders", ArrayType(order_schema))
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

orders_df = parsed_df.select(
    "data.id",
    "data.name",
    explode("data.orders").alias("order")
)

orders_df.select(
    "id",
    "name",
    "order.order_id",
    "order.product",
    "order.quantity",
    "order.price"
).show(truncate=False)


+---+-------+--------+--------+--------+-----+
|id |name   |order_id|product |quantity|price|
+---+-------+--------+--------+--------+-----+
|101|Ahmed  |1       |Laptop  |1       |1000 |
|101|Ahmed  |2       |Mouse   |2       |50   |
|102|Youssef|3       |Keyboard|1       |100  |
+---+-------+--------+--------+--------+-----+



In [0]:
# ============================================================
# Task 5: Calculate total spending for each customer.
# ============================================================

orders_df.groupBy(
    "id",
    "name"
).agg(
    sum(
        col("order.quantity") * col("order.price")
    ).alias("total_spending")
).show(truncate=False)

+---+-------+--------------+
|id |name   |total_spending|
+---+-------+--------------+
|101|Ahmed  |1100          |
|102|Youssef|100           |
+---+-------+--------------+



In [0]:
# ============================================================
# Task 6: Parse JSON and filter users whose age is greater than 25.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","age":25}',),
    ('{"id":102,"name":"Youssef","age":28}',),
    ('{"id":103,"name":"Mohamed","age":30}',)
], ["json_string"])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("age", IntegerType())
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select("data.*") \
    .filter(col("data.age") > 25) \
    .show(truncate=False)

+---+-------+---+
|id |name   |age|
+---+-------+---+
|102|Youssef|28 |
|103|Mohamed|30 |
+---+-------+---+



In [0]:
# ============================================================
# Task 7: Parse JSON and calculate a new column from JSON fields.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","price":1000,"quantity":2}',),
    ('{"id":102,"name":"Youssef","price":500,"quantity":3}',)
], ["json_string"])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("price", IntegerType()),
    StructField("quantity", IntegerType())
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select(
    "data.*"
).withColumn(
    "total",
    col("price") * col("quantity")
).show(truncate=False)

+---+-------+-----+--------+-----+
|id |name   |price|quantity|total|
+---+-------+-----+--------+-----+
|101|Ahmed  |1000 |2       |2000 |
|102|Youssef|500  |3       |1500 |
+---+-------+-----+--------+-----+



In [0]:
# ============================================================
# Task 8: Parse JSON containing a nested address and create a full address.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","address":{"street":"Tahrir","city":"Cairo","country":"Egypt"}}',),
    ('{"id":102,"name":"Youssef","address":{"street":"Faisal","city":"Giza","country":"Egypt"}}',)
], ["json_string"])

address_schema = StructType([
    StructField("street", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType())
])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("address", address_schema)
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select(
    "data.id",
    "data.name",
    concat_ws(
        ", ",
        "data.address.street",
        "data.address.city",
        "data.address.country"
    ).alias("full_address")
).show(truncate=False)

+---+-------+--------------------+
|id |name   |full_address        |
+---+-------+--------------------+
|101|Ahmed  |Tahrir, Cairo, Egypt|
|102|Youssef|Faisal, Giza, Egypt |
+---+-------+--------------------+



In [0]:
# ============================================================
# Task 9: Parse JSON containing an array and find the number of elements.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","skills":["Python","SQL","Spark"]}',),
    ('{"id":102,"name":"Youssef","skills":["Python","Kafka"]}',)
], ["json_string"])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("skills", ArrayType(StringType()))
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select(
    "data.id",
    "data.name",
    "data.skills",
    size("data.skills").alias("number_of_skills")
).show(truncate=False)


+---+-------+--------------------+----------------+
|id |name   |skills              |number_of_skills|
+---+-------+--------------------+----------------+
|101|Ahmed  |[Python, SQL, Spark]|3               |
|102|Youssef|[Python, Kafka]     |2               |
+---+-------+--------------------+----------------+



In [0]:
# ============================================================
# Task 10: Parse JSON and check whether a user has a specific skill.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","skills":["Python","SQL","Spark"]}',),
    ('{"id":102,"name":"Youssef","skills":["Java","Kafka"]}',),
    ('{"id":103,"name":"Mohamed","skills":["Python","Spark"]}',)
], ["json_string"])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("skills", ArrayType(StringType()))
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select(
    "data.*"
).filter(
    array_contains(col("skills"), "Spark")
).show(truncate=False)


+---+-------+--------------------+
|id |name   |skills              |
+---+-------+--------------------+
|101|Ahmed  |[Python, SQL, Spark]|
|103|Mohamed|[Python, Spark]     |
+---+-------+--------------------+



In [0]:
# ============================================================
# Task 11: Parse JSON containing an array of orders and calculate
# the total spending for every order.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","orders":[{"order_id":1,"product":"Laptop","quantity":2,"price":1000},{"order_id":2,"product":"Mouse","quantity":3,"price":50}]}',),
    ('{"id":102,"name":"Youssef","orders":[{"order_id":3,"product":"Keyboard","quantity":2,"price":100}]}',)
], ["json_string"])

order_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("product", StringType()),
    StructField("quantity", IntegerType()),
    StructField("price", IntegerType())
])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("orders", ArrayType(order_schema))
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

orders_df = parsed_df.select(
    "data.id",
    "data.name",
    explode("data.orders").alias("order")
)

orders_df.select(
    "id",
    "name",
    "order.order_id",
    "order.product",
    "order.quantity",
    "order.price",
    (
        col("order.quantity") * col("order.price")
    ).alias("total")
).show(truncate=False)


+---+-------+--------+--------+--------+-----+-----+
|id |name   |order_id|product |quantity|price|total|
+---+-------+--------+--------+--------+-----+-----+
|101|Ahmed  |1       |Laptop  |2       |1000 |2000 |
|101|Ahmed  |2       |Mouse   |3       |50   |150  |
|102|Youssef|3       |Keyboard|2       |100  |200  |
+---+-------+--------+--------+--------+-----+-----+



In [0]:

# ============================================================
# Task 12: Parse JSON containing orders and find the most expensive order.
# ============================================================

orders_df.select(
    "id",
    "name",
    "order.order_id",
    "order.product",
    "order.quantity",
    "order.price",
    (
        col("order.quantity") * col("order.price")
    ).alias("total")
).orderBy(
    col("total").desc()
).show(truncate=False)

+---+-------+--------+--------+--------+-----+-----+
|id |name   |order_id|product |quantity|price|total|
+---+-------+--------+--------+--------+-----+-----+
|101|Ahmed  |1       |Laptop  |2       |1000 |2000 |
|102|Youssef|3       |Keyboard|2       |100  |200  |
|101|Ahmed  |2       |Mouse   |3       |50   |150  |
+---+-------+--------+--------+--------+-----+-----+



In [0]:
# ============================================================
# Task 13: Parse JSON containing orders and find total spending per user.
# ============================================================

orders_df.groupBy(
    "id",
    "name"
).agg(
    sum(
        col("order.quantity") * col("order.price")
    ).alias("total_spending")
).orderBy(
    col("total_spending").desc()
).show(truncate=False)

+---+-------+--------------+
|id |name   |total_spending|
+---+-------+--------------+
|101|Ahmed  |2150          |
|102|Youssef|200           |
+---+-------+--------------+



In [0]:
# ============================================================
# Task 14: Parse JSON containing orders and find users
# who spent more than 1000.
# ============================================================

orders_df.groupBy(
    "id",
    "name"
).agg(
    sum(
        col("order.quantity") * col("order.price")
    ).alias("total_spending")
).filter(
    col("total_spending") > 1000
).show(truncate=False)

+---+-----+--------------+
|id |name |total_spending|
+---+-----+--------------+
|101|Ahmed|2150          |
+---+-----+--------------+



In [0]:
# ============================================================
# Task 15: Parse JSON containing an array of products and
# explode the array into separate rows.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","products":["Laptop","Mouse","Keyboard"]}',),
    ('{"id":102,"name":"Youssef","products":["Monitor","Mouse"]}',)
], ["json_string"])

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("products", ArrayType(StringType()))
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select(
    "data.id",
    "data.name",
    explode("data.products").alias("product")
).show(truncate=False)


+---+-------+--------+
|id |name   |product |
+---+-------+--------+
|101|Ahmed  |Laptop  |
|101|Ahmed  |Mouse   |
|101|Ahmed  |Keyboard|
|102|Youssef|Monitor |
|102|Youssef|Mouse   |
+---+-------+--------+



In [0]:

# ============================================================
# Task 16: Parse JSON with nullable fields and handle missing values.
# ============================================================

json_df = spark.createDataFrame([
    ('{"id":101,"name":"Ahmed","age":25}',),
    ('{"id":102,"name":"Youssef"}',),
    ('{"id":103,"name":"Mohamed","age":30}',)
], ["json_string"])

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

parsed_df = json_df.withColumn(
    "data",
    from_json(col("json_string"), schema)
)

parsed_df.select(
    "data.*"
).fillna(
    {"age": 0}
).show(truncate=False)

+---+-------+---+
|id |name   |age|
+---+-------+---+
|101|Ahmed  |25 |
|102|Youssef|0  |
|103|Mohamed|30 |
+---+-------+---+



In [0]:
# ============================================================
# Task 17: Parse JSON and rename the extracted fields.
# ============================================================

parsed_df.select(
    col("data.id").alias("user_id"),
    col("data.name").alias("user_name"),
    col("data.age").alias("user_age")
).show(truncate=False)


# ============================================================
# Task 18: Parse JSON and sort users by age descending.
# ============================================================

parsed_df.select(
    "data.*"
).orderBy(
    col("age").desc()
).show(truncate=False)


+-------+---------+--------+
|user_id|user_name|user_age|
+-------+---------+--------+
|101    |Ahmed    |25      |
|102    |Youssef  |NULL    |
|103    |Mohamed  |30      |
+-------+---------+--------+

+---+-------+----+
|id |name   |age |
+---+-------+----+
|103|Mohamed|30  |
|101|Ahmed  |25  |
|102|Youssef|NULL|
+---+-------+----+



In [0]:
# ============================================================
# Task 19: Parse JSON and count users by age.
# ============================================================

parsed_df.select(
    "data.*"
).groupBy(
    "age"
).count().orderBy(
    col("age")
).show()

+----+-----+
| age|count|
+----+-----+
|NULL|    1|
|  25|    1|
|  30|    1|
+----+-----+



In [0]:
# ============================================================
# Task 20: Parse JSON containing nested orders and calculate
# the average order value for each user.
# ============================================================

orders_df.groupBy(
    "id",
    "name"
).agg(
    avg(
        col("order.quantity") * col("order.price")
    ).alias("average_order_value")
).show(truncate=False)

+---+-------+-------------------+
|id |name   |average_order_value|
+---+-------+-------------------+
|101|Ahmed  |1075.0             |
|102|Youssef|200.0              |
+---+-------+-------------------+



# `selectExpr()` Function
#### use SQL expression instead of select

In [0]:
df2.show(truncate=False)

+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+--------------------------------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |name_upper    |na

In [0]:
df2.selectExpr(
    "full_name",
    'age',
    "salary",
    "salary * 12 AS yearly_salary"    
).show(10, truncate=False)

+--------------+---+------+-------------+
|full_name     |age|salary|yearly_salary|
+--------------+---+------+-------------+
|Ahmed Ali     |25 |8500  |102000       |
|Youssef Hassan|28 |12000 |144000       |
|Mona Mohamed  |22 |6500  |78000        |
|Omar Khaled   |31 |15000 |180000       |
|Sara Ahmed    |27 |9500  |114000       |
|Mahmoud Adel  |35 |11000 |132000       |
|Nour Ibrahim  |24 |7800  |93600        |
|Karim Samir   |29 |13500 |162000       |
|Hana Mostafa  |26 |7200  |86400        |
|Tarek Emad    |33 |16000 |192000       |
+--------------+---+------+-------------+
only showing top 10 rows


In [0]:
df2.selectExpr(
    "full_name",
    "salary",
    "department",
    "city"
).filter(
    "salary > 10000"
).orderBy(col("salary").desc())\
    .show(truncate=False)

+--------------+------+--------------------+----------+
|full_name     |salary|department          |city      |
+--------------+------+--------------------+----------+
|Khaled Mahmoud|17000 |Data Engineering    |Alexandria|
|Tarek Emad    |16000 |Data Engineering    |Cairo     |
|Omar Khaled   |15000 |Data Engineering    |Cairo     |
|Hossam Fathy  |14500 |Data Engineering    |Giza      |
|Karim Samir   |13500 |Software Engineering|Cairo     |
|Mostafa Nabil |13000 |Software Engineering|Cairo     |
|Amr Sherif    |12500 |IT                  |Giza      |
|Youssef Hassan|12000 |Data Engineering    |Giza      |
|Mahmoud Adel  |11000 |IT                  |Cairo     |
|Ali Hassan    |10500 |IT                  |Cairo     |
+--------------+------+--------------------+----------+



In [0]:
# Repartition
df3 = df2.repartition(4)

# Coalesce
df4 = df2.coalesce(4)

In [0]:
# rank employees by salary 
window_spec = Window.orderBy(col("salary").desc())

df2 = df2.withColumn(
    "rank_by_salary",
    rank().over(window_spec)
)

df2.show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+--------------------------------------+--------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|salary_category|source    |na

In [0]:
# rank employees inside each department
window_spec = Window.partitionBy("department").orderBy(col('salary').desc())

df2 = df2.withColumn(
    "rank_by_salary_in_department",
    rank().over(window_spec)
)

df2.show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+--------------------------------------+--------------+----------------------------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|age_category|

In [0]:
# row number 
window_spec = Window.orderBy(col("salary").desc())

df2 = df2.withColumn(
    "row_number",
    row_number().over(window_spec)
)

df2.show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+--------------------------------------+--------------+----------------------------+----------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_salary|ag

# Conditional Transformation

In [0]:
df2 = df2.withColumn(
    "level",
    when(col("salary") >= 15000, "senior")
    .when(col("salary") >= 10000, "mid")
    .otherwise("junior")
)

df2.show(truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------------------------------------+---+----------+-------+---------------------+--------------------------+------+---------+----------+--------------+-------------------------------------------------------------------------------+-------------+------+------------------------------------+-------+----------+---------+---------+--------+-------------+------------+---------------+----------+--------------+--------------+-----------+------------------------+--------------+---------+----------+--------------------+--------------------+--------------------------------------+--------------+----------------------------+----------+------+
|address                                   |age|city      |country|department           |email                     |gender|is_active|join_date |full_name     |orders                                                                         |phone        |salary|skills                              |user_id|firts_name|last_name|firstName|lastName|yearly_sa

# Save the output

In [0]:
# save the output to JSON
df2.write \
    .mode("overwrite") \
    .json("/Volumes/workspace/streaming/streaming/JSON/output_json")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df2.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/streaming/streaming/Parquet/output_parquet")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
